<br/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="left"/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="right"/>
<div align="center">
<h2>Bootcamp Data Science — Módulo 2</h2><br/>
<h1>Semana 8 · Martes — LightGBM</h1>
<h3>El gradient boosting de Microsoft, rápido y potente</h3>
<br/>
    <b>Instructor:</b> Jesús Ortiz · jesus.jeduardo7@gmail.com<br/><br/>
    <b>SkillNest · b2b-sonda-data-science</b>
</div>
<br/>

## Objetivos

Al final de la clase van a poder:

1. Entender qué es el gradient boosting y por qué LightGBM lo hace mejor que sklearn.
2. Diferenciar LightGBM de Random Forest (a pesar de que ambos usan árboles).
3. Dominar los hiperparámetros clave: num_leaves, learning_rate, n_estimators, max_depth.
4. Usar early stopping para que el modelo se detenga solo.
5. Combinar LightGBM con Optuna (lo de ayer) para un tuning profesional.
6. Resolver UN ejercicio largo donde LightGBM compita contra Random Forest y Logistic Regression.

# 1. Boosting vs Bagging

Hasta acá conocíamos los ensembles tipo **Bagging** (Random Forest): muchos árboles entrenados en paralelo sobre muestras aleatorias, después se promedia. Cada árbol es independiente.

**Boosting** es distinto: los árboles se entrenan **en secuencia**, donde cada árbol nuevo trata de corregir los errores del anterior. La analogía: es como tener un equipo de estudiantes resolviendo un examen donde cada uno se enfoca en las preguntas que los anteriores fallaron.

| | Bagging (RF) | Boosting (LightGBM, XGBoost) |
|---|---|---|
| Cómo entrena | Árboles en paralelo | Árboles en secuencia |
| Cada árbol corrige... | Nada, son independientes | Errores del árbol anterior |
| Velocidad | Rápido (paraleliza fácil) | Más lento (secuencial) |
| Overfitting | Más resistente | Más sensible, hay que cuidar |
| Suele ganar en... | Datasets simples | Competencias Kaggle, datasets tabulares grandes |

Si hay un modelo que gana Kaggle en problemas tabulares, es alguna variante de boosting: XGBoost, LightGBM o CatBoost. Por eso vale la pena conocerlos.

# 2. Por qué LightGBM es "light"

Microsoft sacó LightGBM en 2017 con dos optimizaciones clave que lo hacen entre 10 y 20 veces más rápido que XGBoost:

1. **Histogram-based**: en vez de probar cada valor único de cada feature, agrupa valores en bins (256 por defecto). Pierde precisión teórica pero acelera muchísimo y casi no afecta el resultado.

2. **Leaf-wise growth**: los otros boosters crecen el árbol nivel por nivel (level-wise). LightGBM crece por hoja: en cada paso elige la hoja que reducirá más el error. Es más eficiente.

El resultado: mismo rendimiento que XGBoost pero MUCHO más rápido. En la práctica, si tu dataset tiene más de 10k filas, usá LightGBM.

Instalación: `pip install lightgbm`

# 3. Hiperparámetros clave

LightGBM tiene como 50 hiperparámetros, pero estos son los que importan en el 90% de los casos:

| Hiperparámetro | Qué hace | Default | Rango típico |
|---|---|---|---|
| `n_estimators` | Número de árboles | 100 | 100-1000 |
| `learning_rate` | Cuánto contribuye cada árbol nuevo | 0.1 | 0.01-0.3 |
| `num_leaves` | Hojas máximas por árbol | 31 | 15-127 |
| `max_depth` | Profundidad máxima (-1 = sin límite) | -1 | 3-12 |
| `min_child_samples` | Muestras mínimas por hoja | 20 | 5-100 |
| `reg_alpha` | Regularización L1 | 0 | 0-10 |
| `reg_lambda` | Regularización L2 | 0 | 0-10 |
| `subsample` | % de filas a usar por árbol | 1.0 | 0.6-1.0 |
| `colsample_bytree` | % de features a usar por árbol | 1.0 | 0.6-1.0 |

La regla más importante: si subís `n_estimators`, bajá `learning_rate` (y viceversa). Más árboles con paso chico es mejor que pocos árboles con paso grande.

## Setup

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import lightgbm as lgb

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

# 4. LightGBM básico

In [ ]:
modelo_lgb = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1
)

t0 = time.time()
modelo_lgb.fit(X_train, y_train)
t_lgb = time.time() - t0

pred = modelo_lgb.predict(X_test)
print(f'R²:   {r2_score(y_test, pred):.4f}')
print(f'MAE:  {mean_absolute_error(y_test, pred):.4f}')
print(f'Tiempo: {t_lgb:.2f}s')

# 5. Early stopping: que el modelo decida cuántos árboles necesita

Una de las ventajas de LightGBM es el **early stopping**. Le decimos: entrena hasta n_estimators, pero si después de M rondas no mejoras en validation, parate.

Esto:
- Evita overfitting (no sigues entrenando árboles cuando ya no mejoras).
- Ahorra tiempo (no entrenas árboles que sobran).
- Es básicamente magia.

In [ ]:
# Hacemos un split adicional para validation
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

modelo_es = lgb.LGBMRegressor(
    n_estimators=2000,  # Le decimos que puede entrenar hasta 2000
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1
)
modelo_es.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

print(f'\nÁrboles que realmente entrenó (best_iteration): {modelo_es.best_iteration_}')
print(f'R² test: {modelo_es.score(X_test, y_test):.4f}')
print(f'\nVean que aunque le dije 2000, paró antes porque dejó de mejorar en validation.')

# 6. Feature Importance

Igual que Random Forest, LightGBM da importancia de features. Pero ofrece DOS tipos:

- `importance_type='gain'`: cuánto reduce el error cada feature (la mejor para interpretar).
- `importance_type='split'`: cuántas veces se usó cada feature (la más fácil de hacer pero menos informativa).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
lgb.plot_importance(modelo_lgb, max_num_features=10, importance_type='gain',
                     ax=axes[0], title='Importance por GAIN (cuánto reduce el error)')
lgb.plot_importance(modelo_lgb, max_num_features=10, importance_type='split',
                     ax=axes[1], title='Importance por SPLIT (cuántas veces se usó)')
plt.tight_layout(); plt.show()

# 7. Tuning de LightGBM con Optuna (lo de ayer)

Combinamos lo de ayer con lo de hoy. Optuna busca los mejores hiperparámetros de LightGBM de forma inteligente.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves':       trial.suggest_int('num_leaves', 15, 127),
        'max_depth':        trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0, 10),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0, 10),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    }
    modelo = lgb.LGBMRegressor(**params, random_state=42, verbose=-1)
    return cross_val_score(modelo, X_train, y_train, cv=3, scoring='r2', n_jobs=-1).mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, show_progress_bar=False)

print(f'Mejor R² CV: {study.best_value:.4f}')
print(f'Mejores params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

modelo_tuneado = lgb.LGBMRegressor(**study.best_params, random_state=42, verbose=-1).fit(X_train, y_train)
print(f'\nR² test: {modelo_tuneado.score(X_test, y_test):.4f}')

---
# Ejercicio integrador

Un solo ejercicio, pero exigente. Comparen LightGBM contra los modelos que ya conocen.

## Reto: predecir el precio de casas comparando 4 enfoques

Dataset: `fetch_california_housing`.

Compitiendo: Ridge, RandomForest, LightGBM básico, LightGBM tuneado con Optuna.

### Lo que tienen que hacer

**Parte A — 3 baselines (15 min)**
1. Entrenar 3 modelos con parámetros por defecto: Ridge, RandomForestRegressor, LGBMRegressor.
2. Reportar R² test, MAE test, RMSE test, tiempo de entrenamiento para cada uno.
3. Hacer una tabla comparativa.

**Parte B — LightGBM con early stopping (10 min)**
4. Hacer un split adicional train/val.
5. Entrenar LightGBM con n_estimators=2000 y early_stopping_rounds=50.
6. Reportar best_iteration y métricas en test.
7. ¿Cuántos árboles entrenó al final? ¿Mejoró respecto al baseline de LightGBM?

**Parte C — LightGBM tuneado con Optuna (25 min)**
8. Definir un objective con los 9 hiperparámetros principales.
9. Correr 30 trials.
10. Reportar mejores hiperparámetros + métricas en test.
11. Graficar `optuna.visualization.plot_param_importances(study)` para ver qué hiperparámetros pesan más.
12. Graficar `optuna.visualization.plot_optimization_history(study)` para ver cómo mejoró el modelo.

**Parte D — Análisis del ganador (15 min)**
13. Comparar los 4 modelos en una tabla final (R², MAE, RMSE, tiempo).
14. Graficar Feature Importance del LightGBM tuneado (con `importance_type='gain'`).
15. Graficar real vs predicho del modelo ganador.
16. ¿Cuál ganó? ¿La mejora justifica el tiempo extra del tuning?
17. ¿Qué variables explican el precio de las casas según el modelo?

**Parte E — Decisión final**
18. Si tuvieran que entregar UN modelo a producción para una inmobiliaria, ¿cuál sería? Justifiquen pensando en: precisión, velocidad de entrenamiento, velocidad de predicción y explicabilidad.

### Bonus que vale puntos extra

- Probar LightGBM con `boosting_type='dart'` (otra variante de boosting): ¿cambia el rendimiento?
- Subir n_trials de Optuna a 100: ¿cuánto mejora el R²? ¿Vale la pena el tiempo?
- Combinar early stopping con Optuna: en el objective, usar fit con eval_set y early_stopping_rounds.

In [ ]:
# Parte A — 3 baselines



In [ ]:
# Parte B — LightGBM con early stopping



In [ ]:
# Parte C — LightGBM tuneado con Optuna



In [ ]:
# Parte D — Análisis del ganador



In [ ]:
# Parte E — Decisión final



## Cierre

Hoy aprendimos:

- La diferencia entre Bagging (paralelo, RF) y Boosting (secuencial, LightGBM).
- Por qué LightGBM es 10-20x más rápido que XGBoost (histogram-based + leaf-wise).
- Los hiperparámetros que realmente importan (n_estimators, learning_rate, num_leaves, etc).
- Early stopping para evitar overfitting y ahorrar tiempo.
- Cómo combinar LightGBM con Optuna para un tuning profesional.

Mañana cerramos con XGBoost, que es el hermano más antiguo y aún relevante de LightGBM.